# Smart-Shelf · 06 · Build Supply Chain Simulator (xlsx)

**Deliverable 2.** Excel-based simulator that quantifies margin uplift from reducing Lead Time. Three loss mechanisms modelled:\n1. Late-delivery refunds\n2. Stockout-cancellation revenue loss\n3. Customer-churn LTV loss\n\nAll baselines pulled from the real Olist data (script 01 master table).

## Setup — auto-resolving paths

Run this cell first. It finds the project root automatically.

In [ ]:


from pathlib import Path
import os

# Find smart_shelf root by walking up from current working directory
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "scripts").exists() and (parent / "outputs").exists():
            return parent
        if parent.name == "smart_shelf":
            return parent
    # Fallback: assume notebook lives in smart_shelf/notebooks/
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"   # sibling of smart_shelf/
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

assert DATASET_DIR.exists(), f"Dataset folder not found at {DATASET_DIR}. Place Olist CSVs there."


In [ ]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, Reference
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

# Pull real baselines
m = pd.read_parquet(DATA_DIR / "master_orders.parquet")
m_d = m[m["order_status"] == "delivered"].copy()

baseline = {
    "total_orders"          : int(len(m)),
    "delivered_orders"      : int(len(m_d)),
    "avg_revenue_per_order" : float(m_d["price"].mean()),
    "avg_freight_per_order" : float(m_d["freight_value"].mean()),
    "current_lead_time_d"   : float(m_d["actual_lead_time_days"].mean()),
    "current_late_rate"     : float(m_d["is_late"].mean()),
    "current_stockout_rate" : float(m_d["stockout_risk_flag"].mean()),
    "low_review_rate"       : float((m_d["review_score"].fillna(3) <= 2).mean()),
}

print("Real baselines computed from Olist data:")
for k, v in baseline.items():
    print(f"  {k:<25} = {v:,.4f}" if isinstance(v, float) else f"  {k:<25} = {v:,}")


## Build the workbook\n\nFour sheets: README, Baselines (real data), Simulator (interactive), Scenarios (1-5 day grid).

In [ ]:
wb = Workbook()
ARIAL = lambda **kw: Font(name="Arial", **kw)
header_fill = PatternFill("solid", fgColor="1F3864")
input_fill  = PatternFill("solid", fgColor="FFF2CC")
band_fill   = PatternFill("solid", fgColor="F2F2F2")
result_fill = PatternFill("solid", fgColor="E2EFDA")
thin = Side(border_style="thin", color="BFBFBF")
box  = Border(left=thin, right=thin, top=thin, bottom=thin)

# === Sheet 1: README ===
readme = wb.active
readme.title = "README"
readme["A1"] = "Smart-Shelf Supply Chain Simulator"
readme["A1"].font = ARIAL(size=18, bold=True, color="1F3864")
readme["A2"] = "OmniStore — Lead Time → Margin Impact Model"
readme["A2"].font = ARIAL(size=11, italic=True, color="595959")

readme["A4"] = "PURPOSE"; readme["A4"].font = ARIAL(size=12, bold=True)
readme["A5"] = ("Quantify the margin uplift from reducing average Lead Time. "
                "Three loss mechanisms are modeled, each calibrated to the "
                "Olist baseline numbers extracted from script 01.")

readme["A7"] = "HOW TO USE"; readme["A7"].font = ARIAL(size=12, bold=True)
steps = [
    "1. Open the 'Simulator' sheet.",
    "2. Edit the YELLOW cell 'Lead Time reduction (days)' — try 1, 2, 3 …",
    "3. All BLACK formulas recalculate automatically.",
    "4. The 'Total Margin Uplift' cell shows the bottom-line impact.",
    "5. The 'Scenarios' sheet projects 1-5 day reductions side by side.",
    "6. The 'Baselines' sheet shows the source numbers from Olist data."
]
for i, s in enumerate(steps, start=8):
    readme[f"A{i}"] = s

readme["A15"] = "DATA SOURCE"; readme["A15"].font = ARIAL(size=12, bold=True)
readme["A16"] = (f"Olist Brazilian E-commerce dataset · "
                 f"{baseline['delivered_orders']:,} delivered orders analysed.")
readme.column_dimensions["A"].width = 90

print("README sheet built")


In [ ]:
# === Sheet 2: Baselines ===
base = wb.create_sheet("Baselines")
base["A1"] = "BASELINES — Sourced from Olist data via script 01"
base["A1"].font = ARIAL(size=14, bold=True, color="1F3864")
base.merge_cells("A1:C1")

rows = [
    ("Metric", "Value", "Source"),
    ("Total orders analysed",          baseline["total_orders"], "Olist orders table"),
    ("Delivered orders",               baseline["delivered_orders"], "Olist orders table (status=delivered)"),
    ("Avg revenue per order (R$)",     baseline["avg_revenue_per_order"], "Mean of price column"),
    ("Avg freight per order (R$)",     baseline["avg_freight_per_order"], "Mean of freight_value column"),
    ("Current avg lead time (days)",   baseline["current_lead_time_d"], "purchase → delivered timestamp"),
    ("Current late-delivery rate",     baseline["current_late_rate"], "actual > estimated delivery date"),
    ("Current stockout-risk rate",     baseline["current_stockout_rate"], "carrier handoff > shipping_limit_date"),
    ("Low-review rate (≤ 2 stars)",    baseline["low_review_rate"], "Olist reviews table"),
]
for i, r in enumerate(rows, start=3):
    for j, v in enumerate(r):
        c = base.cell(row=i, column=j+1, value=v)
        if i == 3:
            c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill
        else:
            c.font = ARIAL(size=10)
        c.border = box
        if j == 1 and i > 3:
            if "rate" in str(r[0]).lower() or "low-review" in str(r[0]).lower():
                c.number_format = "0.00%"
            elif "R$" in str(r[0]):
                c.number_format = "R$ #,##0.00"
            else:
                c.number_format = "#,##0.00"

base.column_dimensions["A"].width = 36
base.column_dimensions["B"].width = 18
base.column_dimensions["C"].width = 50

print("Baselines sheet built")


In [ ]:
# === Sheet 3: Simulator ===
sim = wb.create_sheet("Simulator")
sim["A1"] = "SUPPLY CHAIN SIMULATOR · Lead Time → Margin"
sim["A1"].font = ARIAL(size=16, bold=True, color="FFFFFF")
sim["A1"].fill = header_fill
sim["A1"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
sim.row_dimensions[1].height = 26
sim.merge_cells("A1:E1")

# INPUTS
sim["A3"] = "INPUTS"
sim["A3"].font = ARIAL(size=12, bold=True, color="1F3864")

inputs = [
    ("Lead Time reduction (days)",                 2.0, "0.0"),
    ("Late-rate elasticity (per day reduced)",    -0.005, "0.00%"),
    ("Stockout-rate elasticity (per day reduced)", -0.006, "0.00%"),
    ("Avg refund/discount on a late order",        0.10, "0.0%"),
    ("Avg revenue lost per stockout (% of order)", 1.00, "0.0%"),
    ("Customer LTV multiplier (low-review penalty)", 0.30, "0.0%"),
]
for i, (lbl, val, fmt) in enumerate(inputs, start=4):
    sim[f"A{i}"] = lbl
    sim[f"B{i}"] = val
    sim[f"A{i}"].font = ARIAL(size=10)
    sim[f"B{i}"].font = ARIAL(size=10, bold=True, color="0000FF")
    sim[f"B{i}"].fill = input_fill
    sim[f"B{i}"].border = box
    sim[f"B{i}"].number_format = fmt

sim["B5"].comment = Comment(
    "Late-rate elasticity: each 1-day lead-time reduction is assumed to drop the late-delivery rate by this much.",
    "Smart-Shelf")
sim["B6"].comment = Comment(
    "Stockout elasticity: calibrated from script 01 finding that stockout rises from 5% to 30% across delay buckets.",
    "Smart-Shelf")

# LINKS TO BASELINES
sim["D3"] = "BASELINE (linked from 'Baselines')"
sim["D3"].font = ARIAL(size=12, bold=True, color="1F3864")

links = [
    ("Delivered orders",            "=Baselines!B5"),
    ("Avg revenue per order (R$)",  "=Baselines!B6"),
    ("Avg freight per order (R$)",  "=Baselines!B7"),
    ("Current lead time (days)",    "=Baselines!B8"),
    ("Current late-rate",           "=Baselines!B9"),
    ("Current stockout-rate",       "=Baselines!B10"),
]
for i, (lbl, formula) in enumerate(links, start=4):
    sim[f"D{i}"] = lbl
    sim[f"E{i}"] = formula
    sim[f"D{i}"].font = ARIAL(size=10)
    sim[f"E{i}"].font = ARIAL(size=10, bold=True, color="008000")
    sim[f"E{i}"].border = box
    if "rate" in lbl: sim[f"E{i}"].number_format = "0.00%"
    elif "R$" in lbl: sim[f"E{i}"].number_format = "R$ #,##0.00"
    elif "lead time" in lbl.lower(): sim[f"E{i}"].number_format = "0.00"
    else: sim[f"E{i}"].number_format = "#,##0"

print("Simulator inputs + baseline links built")


In [ ]:
# CALCULATION BLOCK — explicit row numbers prevent off-by-one bugs
sim["A12"] = "MARGIN-UPLIFT CALCULATION"
sim["A12"].font = ARIAL(size=12, bold=True, color="1F3864")

block = {
    13: ("New lead time (days)",                           "=E7-B4"),
    14: ("New late-delivery rate",                         "=MAX(0,E8+B5*B4)"),
    15: ("New stockout-risk rate",                         "=MAX(0,E9+B6*B4)"),
    17: ("--- LOSS MECHANISM 1 — late-order refunds ---",  ""),
    18: ("Revenue exposed (delivered orders × avg revenue)","=E4*E5"),
    19: ("Current refund cost (R$)",                       "=B18*E8*B7"),
    20: ("New refund cost (R$)",                           "=B18*B14*B7"),
    21: ("Saving 1 — refund reduction (R$)",               "=B19-B20"),
    23: ("--- LOSS MECHANISM 2 — stockout cancellations ---", ""),
    24: ("Current stockout cost (R$)",                     "=B18*E9*B8"),
    25: ("New stockout cost (R$)",                         "=B18*B15*B8"),
    26: ("Saving 2 — stockout reduction (R$)",             "=B24-B25"),
    28: ("--- LOSS MECHANISM 3 — churn / lifetime value ---", ""),
    29: ("Customers exposed (orders × low-review rate)",   "=E4*Baselines!B11"),
    30: ("Lost LTV at current late-rate (R$)",             "=B29*E8*E5*B9"),
    31: ("Lost LTV at new late-rate (R$)",                 "=B29*B14*E5*B9"),
    32: ("Saving 3 — LTV protection (R$)",                 "=B30-B31"),
}
for r, (lbl, formula) in block.items():
    sim[f"A{r}"] = lbl
    if formula:
        sim[f"B{r}"] = formula
        sim[f"B{r}"].font = ARIAL(size=10)
    if "rate" in lbl: sim[f"B{r}"].number_format = "0.00%"
    elif "lead time" in lbl: sim[f"B{r}"].number_format = "0.00"
    elif "Customers" in lbl: sim[f"B{r}"].number_format = "#,##0"
    elif "R$" in lbl or "Saving" in lbl or "cost" in lbl or "exposed" in lbl.lower():
        sim[f"B{r}"].number_format = "R$ #,##0"
    if "---" in lbl:
        sim[f"A{r}"].font = ARIAL(size=10, bold=True, italic=True, color="595959")
    elif "Saving" in lbl:
        sim[f"A{r}"].font = ARIAL(size=10, bold=True)
        sim[f"B{r}"].font = ARIAL(size=10, bold=True, color="00703C")
        sim[f"B{r}"].fill = result_fill

# TOTAL UPLIFT
sim["A34"] = "TOTAL MARGIN UPLIFT"
sim["A34"].font = ARIAL(size=14, bold=True, color="FFFFFF")
sim["A34"].fill = header_fill
sim["A34"].alignment = Alignment(horizontal="left", indent=1)
sim["B34"] = "=B21+B26+B32"
sim["B34"].font = ARIAL(size=14, bold=True, color="FFFFFF")
sim["B34"].fill = header_fill
sim["B34"].number_format = "R$ #,##0"
sim["B34"].alignment = Alignment(horizontal="right", indent=1)

sim["A35"] = "Margin uplift per day reduced"
sim["A35"].font = ARIAL(size=11, bold=True)
sim["B35"] = "=IFERROR(B34/B4,0)"
sim["B35"].number_format = "R$ #,##0"
sim["B35"].font = ARIAL(size=11, bold=True, color="00703C")

sim["A36"] = "Margin uplift as % of total revenue"
sim["A36"].font = ARIAL(size=11, bold=True)
sim["B36"] = "=IFERROR(B34/(E4*E5),0)"
sim["B36"].number_format = "0.00%"
sim["B36"].font = ARIAL(size=11, bold=True, color="00703C")

sim.column_dimensions["A"].width = 50
sim.column_dimensions["B"].width = 20
sim.column_dimensions["C"].width = 3
sim.column_dimensions["D"].width = 36
sim.column_dimensions["E"].width = 18

print("Simulator calculation block built")


In [ ]:
# === Sheet 4: Scenarios ===
scen = wb.create_sheet("Scenarios")
scen["A1"] = "SCENARIOS — Lead Time reduction sensitivity"
scen["A1"].font = ARIAL(size=14, bold=True, color="1F3864")
scen.merge_cells("A1:G1")

scen_headers = ["Metric", "0 days (current)", "1 day", "2 days", "3 days", "4 days", "5 days"]
for j, h in enumerate(scen_headers, start=1):
    c = scen.cell(row=3, column=j, value=h)
    c.font = ARIAL(size=10, bold=True, color="FFFFFF"); c.fill = header_fill
    c.alignment = Alignment(horizontal="center", wrap_text=True); c.border = box

metric_labels = [
    "LT reduction (days)", "New late-rate", "New stockout-rate",
    "Saving 1 — refunds (R$)", "Saving 2 — stockouts (R$)",
    "Saving 3 — LTV (R$)", "TOTAL UPLIFT (R$)", "Uplift % of revenue"
]
for i, lbl in enumerate(metric_labels, start=4):
    scen.cell(row=i, column=1, value=lbl).font = ARIAL(size=10, bold=("TOTAL" in lbl))
    if "TOTAL" in lbl:
        scen.cell(row=i, column=1).fill = result_fill

for col_idx, days in enumerate([0, 1, 2, 3, 4, 5], start=2):
    L = days
    scen.cell(row=4, column=col_idx, value=L).number_format = "0"
    scen.cell(row=5, column=col_idx,
              value=f"=MAX(0,Baselines!B9+Simulator!B5*{L})").number_format = "0.00%"
    scen.cell(row=6, column=col_idx,
              value=f"=MAX(0,Baselines!B10+Simulator!B6*{L})").number_format = "0.00%"
    scen.cell(row=7, column=col_idx,
              value=f"=(Baselines!B5*Baselines!B6)*(Baselines!B9 - "
                    f"MAX(0,Baselines!B9+Simulator!B5*{L}))*Simulator!B7"
              ).number_format = "R$ #,##0"
    scen.cell(row=8, column=col_idx,
              value=f"=Baselines!B5*Baselines!B6*(Baselines!B10 - "
                    f"MAX(0,Baselines!B10+Simulator!B6*{L}))*Simulator!B8"
              ).number_format = "R$ #,##0"
    scen.cell(row=9, column=col_idx,
              value=f"=Baselines!B5*Baselines!B11*Baselines!B6*(Baselines!B9 - "
                    f"MAX(0,Baselines!B9+Simulator!B5*{L}))*Simulator!B9"
              ).number_format = "R$ #,##0"
    scen.cell(row=10, column=col_idx,
              value=f"=SUM({get_column_letter(col_idx)}7:{get_column_letter(col_idx)}9)"
              ).number_format = "R$ #,##0"
    scen.cell(row=10, column=col_idx).font = ARIAL(size=10, bold=True, color="00703C")
    scen.cell(row=10, column=col_idx).fill = result_fill
    scen.cell(row=11, column=col_idx,
              value=f"=IFERROR({get_column_letter(col_idx)}10/(Baselines!B5*Baselines!B6),0)"
              ).number_format = "0.00%"

for r in range(3, 12):
    for c in range(1, 8):
        scen.cell(row=r, column=c).border = box

scen.column_dimensions["A"].width = 30
for c in range(2, 8):
    scen.column_dimensions[get_column_letter(c)].width = 16

# Chart
chart = LineChart()
chart.title = "Margin Uplift vs Lead Time Reduction"
chart.x_axis.title = "Days reduced"
chart.y_axis.title = "Total margin uplift (R$)"
data = Reference(scen, min_col=2, min_row=10, max_col=7, max_row=10)
cats = Reference(scen, min_col=2, min_row=4, max_col=7, max_row=4)
chart.add_data(data, titles_from_data=False)
chart.set_categories(cats)
chart.height = 9; chart.width = 18
scen.add_chart(chart, "A14")

# SAVE
out_file = OUTPUTS_DIR / "Supply_Chain_Simulator.xlsx"
wb.save(out_file)
print(f"Saved: {out_file}")
print("\nNext step: open the file in Excel/LibreOffice. Default 2-day reduction should show:")
print("  Total margin uplift ≈ R$ 178,000 (1.34% of revenue)")


✅ **Notebook 06 complete.** Move to `07_build_executive_report.ipynb`.